In [56]:
import pandas as pd
import numpy as np
import math

from datetime import datetime, timedelta, date

import talib

In [57]:
def _print(msg):
    if True:
        print(msg)
        
def pyramid(context, data, stock, df, price_for_stop, quantity_total, quantity_start, price_high_water, price_avg, stop_loss_amount, stop_loss_atr_stds = 20.0, price_target_multiple = 0.75, profit_to_risk = 0.2):
    price = df.iloc[-1]['close']
    timeperiod = 14
#     sym = stock.symbol
    sym = "SYNTH"

    df['rsi'] = talib.RSI(df['close'].to_numpy(), timeperiod = timeperiod)

    last = df.iloc[-1]
    price_current = last['close']

    atr, atr_std = atr_get(df)

    # Nothing to do yet
    if quantity_total == 0:
        return 0, 0.0, 0, 0

    profit_per_share = price_high_water - price_avg
    profit_per_share_current = price_current - price_avg

    # How much room we need to give on the upside
    risk = (atr + (atr_std * price_target_multiple))

    # Increase that room for us to risk some of our profit
    # but make sure that the amount risked has at least 
    # a PT multiple of room to move
    profit_needed = (risk / profit_to_risk)

    profit_after_risk = profit_per_share - profit_needed
    profit_percent = profit_per_share_current / price_avg

    # Figure out stop loss
    if quantity_total == quantity_start:
        # Initial Stop Loss before a scale in
        if stop_loss_amount is False:
            stop_loss_amount = (atr + (atr_std * stop_loss_atr_stds))
        stop_loss = price_avg - stop_loss_amount
        target = price_avg + profit_needed

        #_print("[%s]\tStop Loss: %0.2f\tPrice: %0.2f\tTarget: %0.2f\tprice_avg: %0.2f\tstop_loss_amount: %0.2f\tHigh Water: %0.2f\tProfit: share: %0.2f\tneed: %0.2f\tafter risk: %0.2f\tpercent: %0.2f" % (sym, stop_loss, price_current, target, price_avg, stop_loss_amount, price_high_water, profit_per_share, profit_needed, profit_after_risk, profit_percent*100.0))
    else:
        # After have scaled in, don't want to give up too much profit
        stop_loss_amount = profit_per_share * profit_to_risk
        stop_loss = price_high_water - stop_loss_amount
        target = False
        #_print("[%s]\tUpdate Stop Loss: %0.2f\tPrice: %0.2f\tprice_avg: %0.2f\ttarget/price_high_water: %0.2f\tstop_loss_amount: %0.2f profit percent: %0.2f" % (sym, stop_loss, price_current, price_avg, price_high_water, stop_loss_amount, profit_percent*100.0))

    # Check if stopped out
    if price_for_stop <= stop_loss and price_current <= stop_loss:
        _print("[%s]\tStop Out: %0.2f" % (sym, stop_loss))
        return -quantity_total, min(stop_loss, last['open']), target, stop_loss

    if last['close'] >= target or target == False:
        if profit_after_risk > 0:
            #_print("[%s]\tTarget Hit: Scale in: %0.2f" % (sym, max(target, price_high_water)))
            price_avg_target = price_high_water - profit_needed
            price_delta = price_current - price_avg_target
            shares_need = int(math.ceil((price_current * quantity_total) - (price_avg * quantity_total) - (price_delta * quantity_total)) / (price_current - price_avg_target))

            if shares_need > 0:
                _print("{:<5}\t{:>5.2f}%    Price: {:>7.2f}    Stop Loss: {:>7.2f} ({:>6.2f}%)    Target: {:>7.2f} ({:>6.2f}%)    price_avg: {:>7.2f}    Shares vs Start: {:>4.1f}%    stop_loss_amount: {:>5.2f}    High Water: {:>7.2f}    Profit: per share: {:>5.2f}    need: {:>5.2f}    after risk: {:>6.2f}   RSI: {:>3d}".format(sym, profit_percent*100.0, price_current, stop_loss, ((stop_loss - price_current) / price_current)*100.0, target, ((target - price_current) / price_current)*100.0, price_avg, (quantity_total / quantity_start)*100.0, stop_loss_amount, price_high_water, profit_per_share, profit_needed, profit_after_risk, int(last['rsi'])))
                _print("[%s]\tTarget Hit: Scale In\tPrice: %0.2f\tPrice Target: %0.2f\tPrice Avg: %0.2f\tPrice Delta: %0.2f\tShares Needed: %d" % (sym, price_current, price_avg_target, price_avg, price_delta, shares_need))
                return shares_need, price_current, target, stop_loss

    if target is False:
        target = price_high_water
    _print("{:<5}\t{:>5.2f}%    Price: {:>7.2f}    Stop Loss: {:>7.2f} ({:>6.2f}%)    Target: {:>7.2f} ({:>6.2f}%)    price_avg: {:>7.2f}    Shares vs Start: {:>4.1f}%    stop_loss_amount: {:>5.2f}    High Water: {:>7.2f}    Profit: per share: {:>5.2f}    need: {:>5.2f}    after risk: {:>6.2f}   RSI: {:>3d}".format(sym, profit_percent*100.0, price_current, stop_loss, ((stop_loss - price_current) / price_current)*100.0, target, ((target - price_current) / price_current)*100.0, price_avg, (quantity_total / quantity_start)*100.0, stop_loss_amount, price_high_water, profit_per_share, profit_needed, profit_after_risk, int(last['rsi'])))
    #_print("[%s]\t%0.2f%%\tPrice: %0.2f\tStop Loss: %0.2f (%0.2f%%)\tTarget: %0.2f (%0.2f%%)\tprice_avg: %0.2f\tShares vs Start: %0.2f%%\tstop_loss_amount: %0.2f\tHigh Water: %0.2f\tProfit: per share: %0.2f\tneed: %0.2f\tafter risk: %0.2f RSI: %d" % (sym, profit_percent*100.0, price_current, stop_loss, ((stop_loss - price_current) / price_current)*100.0, target, ((target - price_current) / price_current)*100.0, price_avg, (quantity_total / quantity_start)*100.0, stop_loss_amount, price_high_water, profit_per_share, profit_needed, profit_after_risk, int(last['rsi'])))
    return 0, 0.0, target, stop_loss

def atr_get(hlc_atr):
    timeperiod_atr = 14
    atr = talib.ATR(hlc_atr['high'].to_numpy(), hlc_atr['low'].to_numpy(), hlc_atr['close'].to_numpy(), timeperiod = timeperiod_atr)
    atr = atr[~np.isnan(atr)][-1]
#     atr_std = np.std(atr)
#     atr_std = 0.25
#     atr = max(np.random.normal(1, atr_std), 0.01)

    print("ATR: %0.2f ATR STD: %0.2f" % (atr, atr_std))
    
    return atr, atr_std



In [62]:
context = {}
data = None
stock = None

atr = 1.0
atr_std = 0.25

multiple = 1
lookback = 150
start_price = 100
window = 21
date_start = datetime(2020, 10, 1, 9, 30, 0, 0)

ohlcv = {
    'close': [x for x in range(start_price, lookback * multiple, multiple)],
}

df = pd.DataFrame(ohlcv, index = [date_start + timedelta(days=x) for x in range(start_price, lookback * multiple)])
df['high'] = df['close'] + np.random.normal(1.0, atr_std/2.0)
df['low'] = df['close'] - np.random.normal(1.0, atr_std/2.0)
df['close'] = df['close'] + 0.0

shares_total = 0
shares_start = 20
price_high_water = 0
price_avg = 0.0
stop_loss_amount = False

for i in range(window, len(df)):
    df_tmp = df.iloc[i-lookback:i].dropna(axis=1)
    dt = df_tmp.iloc[-1].name
    last = df_tmp.iloc[-1]['close']
    
    # Kick us off
    if shares_total == 0:
        shares_total = shares_start

    shares_need, price_current, target, stop_loss = pyramid(context, data, stock, df, last, shares_total, shares_start, price_high_water, price_avg, stop_loss_amount, stop_loss_atr_stds = 20.0, price_target_multiple = 0.75, profit_to_risk = 0.2)
    cost_total = shares_total + price_avg
    price_high_water = max(price_high_water, max(df_tmp['high']))

    if shares_need > 0:
        cost_total += shares_need * price_current
        shares_total += shares_need
        price_avg = cost_total / shares_total
    if shares_need < 0:
        cost_total = 0.0
        shares_total = 0
        price_avg = 0.0
        price_high_water = 0.0
        


SYNTH	  inf%    Price:  149.00    Stop Loss:   -6.64 (-104.46%)    Target:    9.14 (-93.86%)    price_avg:    0.00    Shares vs Start: 100.0%    stop_loss_amount:  6.64    High Water:    0.00    Profit: per share:  0.00    need:  9.14    after risk:  -9.14   RSI: 100
SYNTH	  inf%    Price:  149.00    Stop Loss:   -6.01 (-104.03%)    Target:    6.00 (-95.98%)    price_avg:    0.00    Shares vs Start: 100.0%    stop_loss_amount:  6.01    High Water:  120.85    Profit: per share: 120.85    need:  6.00    after risk: 114.86   RSI: 100
[SYNTH]	Target Hit: Scale In	Price: 149.00	Price Target: 114.86	Price Avg: 0.00	Price Delta: 34.14	Shares Needed: 67
SYNTH	29.59%    Price:  149.00    Stop Loss:  120.48 (-19.14%)    Target:    0.00 (-100.00%)    price_avg:  114.98    Shares vs Start: 435.0%    stop_loss_amount:  1.37    High Water:  121.85    Profit: per share:  6.87    need:  6.27    after risk:   0.61   RSI: 100
[SYNTH]	Target Hit: Scale In	Price: 149.00	Price Target: 115.58	Price Avg: 114

/Users/moilanen/Library/Python/3.7/lib/python/site-packages/ipykernel_launcher.py:34: RuntimeWarning: divide by zero encountered in double_scalars
